<img src="ost_logo.png" width="240" height="240" align="right"/>
<div style="text-align: left"> <b> Machine Learning </b> <br> MSE FTP MachLe <br> 
<a href="mailto:christoph.wuersch@ost.ch"> Christoph Würsch </a> </div>

# ML08: A3 Support Vector Machine: Pima Indians
MSE_FTP_MachLe, WÜRC


In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

## Pima-Indian dataset

The data set `pima-indians-diabetes.csv` contains medical indicators collected from 768 patients of the indigenous Pima tribe (Phoenix, AZ). The subjects were between 21 and 81 years old. The following characteristics were recorded:

- `NumTimesPrg`: Number of pregnancies
- `PlGlcConc`: Blood sugar 2h after oral uptake of sugar (oGTT) (mg/dl)
- `BloodP`: Diastolic blood pressure (mm Hg)
- `SkinThick`: thickness of the skin fold at the triceps (mm)
- `TwoHourSerIns`: Insulin concentration after 2h oGTT ( 𝜇 IU/ mg)
- `BMI`:Body Mass Index (kg/m 2 )
- `DiPedFunc`: Hereditary predisposition
- `Age`: Age (years)
- `HasDiabetes` :Diagnosis Diabetes Type II

The classification goal is to make the diagnosis of type II diabetes based on the factors, i.e. to make a prediction model for the variable `y= HasDiabetes` using a support vector machine (SVM) with a radial basis function kernel.

![Pima Indians](women-Pima-shinny-game-field-hockey.jpg)


### Explorative Data Analysis (EDA)

Load the dataset `pima-indians-diabetes.csv` and create a `pandas` data frame from it. We take care that the column captions are imported correctly. 

In [ ]:
pima = pd.read_csv('./pima-indians-diabetes.csv', header=None)
pima.columns = ["NumTimesPrg", "PlGlcConc", "BloodP", "SkinThick", "TwoHourSerIns", "BMI", "DiPedFunc", "Age", "HasDiabetes"]

Display the first 5 entries of the dataset.


In [ ]:
pima.head(5)


It is already noticeable here that the insulin values of some patients have the value '0'. 

### (a) Calculate the percentage of patients with diabetes and display a statistics using `df.describe`
We can calculate the percentage of the patients with diabetes by determining the mean value of the response column 'HasDiabetes'. Using the `dataframe.describe()`, we can print the main statistics of the dataset.

In [ ]:
perc_diab = pima['HasDiabetes'].mean()

print('percentage of diabetes: %f ' % np.round(perc_diab*100, 2))

y = pima['HasDiabetes']
X = pima.drop('HasDiabetes', axis=1)

X_names = X.columns

In [ ]:
pima.describe()

Some characteristics take the value 0, although this makes no medical sense. These are

- `PlGlcConc`
- `BloodP`
- `SkinThick`
- `TwoHourSerins`
- `BMI`

For example, the insulin value has at least a quarter missing. For these characteristics, we replace the 0 values with `np.nan` and then count again. For all other characteristics we do not know whether there are any other missing values.

### (b) Exlude samples with zero entries or missing values

Replace the 0 values with `np.nan`and print again a statstical description of the dataset using `df.describe()`. Then drop `np.nan` values using `df.dropna()`.

In [ ]:
print(X_names.values)

In [ ]:
pima2 = pima.loc[:,X_names.values]
target= pima['HasDiabetes']
pima2.replace(0, np.nan,inplace=True)
pima2['HasDiabetes']=target

pima2.dropna(axis=0,inplace=True)
pima=pima2

In [ ]:
pima.describe()

### (c) Plot a histogram of the of each feature and the the target using `df.hist()`

In [ ]:
plt.figure()
pima.hist(figsize=(12,12),bins=37)
plt.show()

In [ ]:
corrmatrix=pima.corr()
import seaborn as sns
corrmatrix
sns.heatmap(corrmatrix)

### (d) Split the data in 80% training and 20% test data

   Use `train_test_split` from `sklearn.model_selection`. If you feed a `pandas.Dataframe` as an input to the method, you will also get `pandas.Dataframes` as output for the training and test features. This is quite practical.

In [ ]:
from sklearn.model_selection import train_test_split

Xtr, Xtest, ytrain, ytest = train_test_split(pima.loc[:,X_names.values],pima.HasDiabetes,train_size=0.8)

Xtr.head()

### (e) Standardize the features using the `StandardScaler` from `sklearn.preprocessing`
Standardize the features using the `StandardScaler` from `sklearn.preprocessing` and display the histograms of the features again.

In [ ]:
from sklearn.preprocessing import StandardScaler, PowerTransformer

scaler = PowerTransformer()

Xtrain_scale = scaler.fit_transform(Xtr)
Xtest_scale = scaler.transform(Xtest)

Xtrain = pd.DataFrame(Xtrain_scale, columns=X.columns)
Xtrain.hist(bins=50, figsize=(20, 15))
plt.show()

###  (f) Train a support vector machine with a radial basis function kernel and determine the accuracy on the test data.



In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

model=SVC(kernel='rbf', gamma=1, C=1)
model.fit(Xtrain, ytrain)
ypred=model.predict(Xtest_scale)
print('Accuracy: %f' % accuracy_score(ytest, ypred))

### (g) Perform a cross validated grid search `GridSearchCV` to find the best parameters for $\gamma$ and $C$ and determine the best parameters and score
- vary the value of $C$ in the range in a logarithmic scale from $10^{-3}$ to $10^{+3}$ (7 steps)
- vary the value of $\gamma$ in the range in a logarithmic scale from $10^{-3}$ to $10^{+3}$ (7 steps)
- print a classification report and a confusion matrix of the best classifier using `classification_report` and `confusion_matrix` from `sklearn.metrics`.


In [ ]:
C_range = np.logspace(-3, 3, 7)
gamma_range = np.logspace(-3, 3, 7)
param_grid = dict(gamma=gamma_range, C=C_range)

model=SVC()

cv = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
grid = GridSearchCV(model, param_grid=param_grid, cv=cv,n_jobs=-1)
grid.fit(Xtrain, ytrain)

print("The best parameters are %s with a score of %0.2f"
      % (grid.best_params_, grid.best_score_))



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

ypred = grid.best_estimator_.predict(Xtest_scale)
target_names = ['negative', 'positive']
print(classification_report(ytest, ypred, target_names=target_names))
print(confusion_matrix(ytest, ypred))

### (h) Plot the resulting decision boundary of the SVM classifier for different values of $\gamma$ and $C$ as contour plot in 2D
 
Use the following features as the only features for the prediction such that we can display the decision boundary in a 2D plot. 
- feature 1: `TwoHourSerIns` (x1-axis)
- feature 2: `Age` (x2-axis) 

Vary the C parameters in the following ranges:
- `C_range = [1e-2, 1, 1e2]`
- `gamma_range = [1e-1, 1, 1e1]`

You can use the helper function `PlotDecisionBoundary(model, X2D, y)` to plot the decision boundary and the margins of the classifier. 


In [ ]:
print(X_names.values)

Xtrain_2D=Xtrain.loc[:,['TwoHourSerIns','Age']]
Xtrain_2D

In [ ]:
def PlotDecisionBoundary(model, X2D, y):

    gamma=model.gamma
    C=model.C
    x1=X2D.iloc[:,0].values
    x2=X2D.iloc[:,1].values
        
    # evaluate decision function in a grid
    xx, yy = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-2, 2, 200))
    
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # visualize decision function for these parameters
    
    plt.title("gamma=10^%d, C=10^%d" % (np.log10(gamma), np.log10(C)),
            size='medium')

    # visualize parameter's effect on decision function
    plt.pcolormesh(xx, yy, -Z, cmap=plt.cm.RdBu,shading='nearest')
    plt.contour(xx, yy, -Z,[-1, 0, 1])
    plt.scatter(x1, x2, c=y, cmap=plt.cm.RdBu_r, edgecolors='k')
    plt.xlim([-2,2])
    plt.ylim([-2,2])


In [ ]:
# Now we need to fit a classifier for all parameters in the 2d version
# (we use a smaller set of parameters here because it takes a while to train)

C_range = [1e-2, 1, 1e2]
gamma_range = [1e-1, 1, 1e1]
classifiers = []
for C in C_range:
    for gamma in gamma_range:
        clf = SVC(kernel='rbf', degree=2, C=C, gamma=gamma)
        clf.fit(Xtrain_2D.values, ytrain)
        classifiers.append((C, gamma, clf))

In [ ]:
# #############################################################################
# Visualization
#
# draw visualization of parameter effects


for (k, (C, gamma, clf)) in enumerate(classifiers):

    plt.figure(figsize=(8, 8))
    # visualize decision function for these parameters
    # plt.subplot(len(C_range), len(gamma_range), k + 1)
    PlotDecisionBoundary(clf, Xtrain_2D, ytrain)
#plt.savefig('diabetes.pdf')